In [22]:
!apt-get update && apt-get install -y build-essential

!pip install torchmetrics
!pip install cmdstanpy==1.2.5
!pip uninstall prophet -y
!pip install prophet==1.1.4


Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]                
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease                          
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]         
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]            
Get:6 https://cli.github.com/packages stable/main amd64 Packages [356 B]        
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]       
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,479 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [84.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,155 

Found existing installation: prophet 1.3.0
Uninstalling prophet-1.3.0:
  Successfully uninstalled prophet-1.3.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 52.7 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 97.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.2 MB/s eta 0:00:00
  Created wheel for pymeeus: filename=PyMeeus-0.5.12-py3-none-any.whl size=732000 sha256=ed3497226786a5aea0eb73e6f8cb2866cf10742e9f978eee696a84a9934e0c60
  Stored in directory: /root/.cache/pip/wheels/92/74/5d/5cf1193a619dc315b5b0a158876900e21440c71130251307b6
Successfully built pymeeus


In [23]:
import pandas as pd
import numpy as np
import torch
from prophet import Prophet
from torchmetrics import WeightedMeanAbsolutePercentageError
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [24]:
%cd /content/Walmart_sales_forecasting
data_dir = 'data/processed/feature_engineering.feather'
df_feature = pd.read_feather(data_dir)
df_feature

/content/Walmart_sales_forecasting


,Store,Dept,Date,Weekly_Sales,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,...,mean_sales_last_6_week,max_sales_last_6_week,min_sales_last_6_week,std_sales_last_6_week,emw_sales_0.5,emw_sales_0.75,sum_store_1_week,mean_store_1_week,sum_dept_1_week,mean_dept_1_week
6,1,1,2010-03-19,22136.64,A,151315,54.58,2.720,0.00,0.00,...,30360.018333,63593.12,7612.03,26384.953473,20940.430630,15246.700822,1242589.35,17258.185417,846686.47,18815.254889
149,1,2,2010-03-19,43615.49,A,151315,54.58,2.720,0.00,0.00,...,32780.786667,63593.12,7612.03,24477.831239,21538.535315,20414.155206,1242589.35,17258.185417,1742919.72,38731.549333
292,1,3,2010-03-19,9001.37,A,151315,54.58,2.720,0.00,0.00,...,29451.181667,61326.35,7612.03,20480.695326,32577.012657,37815.156301,1242589.35,17258.185417,371295.10,8251.002222
435,1,4,2010-03-19,34118.11,A,151315,54.58,2.720,0.00,0.00,...,20730.351667,43615.49,7612.03,14443.999382,20789.191329,16204.816575,1242589.35,17258.185417,1064848.12,23663.291556
578,1,5,2010-03-19,22632.57,A,151315,54.58,2.720,0.00,0.00,...,25148.031667,43615.49,9001.37,13661.565950,27453.650664,29639.786644,1242589.35,17258.185417,1063601.82,24734.926047
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421012,45,93,2012-10-26,2487.80,B,118221,58.85,3.882,4018.91,58.08,...,17814.415000,54608.75,717.82,20287.128190,35354.118359,45312.322469,760281.43,11347.484030,1049693.34,24992.698571
421146,45,94,2012-10-26,5203.31,B,118221,58.85,3.882,4018.91,58.08,...,18109.411667,54608.75,1689.10,19999.636394,18920.959180,13193.930617,760281.43,11347.484030,1291378.41,29349.509318
421289,45,95,2012-10-26,56017.47,B,118221,58.85,3.882,4018.91,58.08,...,18695.113333,54608.75,2487.80,19466.945450,12062.134590,7200.965154,760281.43,11347.484030,1709432.51,37987.389111
421434,45,97,2012-10-26,6817.48,B,118221,58.85,3.882,4018.91,58.08,...,26666.748333,56017.47,2487.80,23647.747335,34039.802295,43813.343789,760281.43,11347.484030,596563.70,13558.265909


In [25]:
df_feature.info()

<class 'pandas.core.frame.DataFrame'>
Index: 401996 entries, 6 to 421569
Data columns (total 46 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   Store                   401996 non-null  int64         
 1   Dept                    401996 non-null  int64         
 2   Date                    401996 non-null  datetime64[ns]
 3   Weekly_Sales            401996 non-null  float64       
 4   Type                    401996 non-null  object        
 5   Size                    401996 non-null  int64         
 6   Temperature             401996 non-null  float64       
 7   Fuel_Price              401996 non-null  float64       
 8   MarkDown1               401996 non-null  float64       
 9   MarkDown2               401996 non-null  float64       
 10  MarkDown3               401996 non-null  float64       
 11  MarkDown4               401996 non-null  float64       
 12  MarkDown5               401996 non-

In [ ]:
from sklearn.preprocessing import StandardScaler

reg_cols = ['Size', 'Temperature', 'Fuel_Price', 'total_markdown', 'avg_markdown', 'max_markdown']
scaler = StandardScaler()
df_feature[reg_cols] = scaler.fit_transform(df_feature[reg_cols])

In [26]:
test = df_feature[df_feature['is_test']]
train = df_feature[~df_feature['is_test']]

prophet_data = df_feature.groupby(['Date', 'store_dept']).agg(
    {
        'Weekly_Sales' : 'sum',
        'Size' : 'first',
        'Type_encoded' : 'first',
        'IsHoliday_True' : 'first',
        'Temperature': 'first',
        'Fuel_Price' : 'first',
        "total_markdown": "first",   
        "avg_markdown": "first",      
        "max_markdown": "first",
    }
).reset_index()

prophet_data

,Date,store_dept,Weekly_Sales,Size,Type_encoded,IsHoliday_True,Temperature,Fuel_Price,total_markdown,avg_markdown,max_markdown
0,2010-03-19,store_10_dept_1,38252.33,126512,2,False,61.46,3.054,0.00,0.000,0.00
1,2010-03-19,store_10_dept_10,49479.06,126512,2,False,61.46,3.054,0.00,0.000,0.00
2,2010-03-19,store_10_dept_11,30518.98,126512,2,False,61.46,3.054,0.00,0.000,0.00
3,2010-03-19,store_10_dept_12,11762.04,126512,2,False,61.46,3.054,0.00,0.000,0.00
4,2010-03-19,store_10_dept_13,65575.03,126512,2,False,61.46,3.054,0.00,0.000,0.00
...,...,...,...,...,...,...,...,...,...,...,...
401991,2012-10-26,store_9_dept_91,914.84,125833,2,False,69.52,3.506,2189.61,437.922,1666.38
401992,2012-10-26,store_9_dept_92,18310.28,125833,2,False,69.52,3.506,2189.61,437.922,1666.38
401993,2012-10-26,store_9_dept_94,233.02,125833,2,False,69.52,3.506,2189.61,437.922,1666.38
401994,2012-10-26,store_9_dept_95,32382.05,125833,2,False,69.52,3.506,2189.61,437.922,1666.38


In [ ]:
def build_prophet_model(prophet_data,  test_data):
    min_test_time = test_data['Date'].min()
    max_test_time = test_data['Date'].max()
    
    prophet_model = {}
    prophet_predictions = {}
    prophet_metrics = pd.DataFrame(columns=['combo', 'mae', 'rmae', 'wape'])
    all_real = []
    all_predict = [] 
    
    
    for combination in prophet_data['store_dept'].unique():
        print(f'Build prophet model for {combination}')
        
        combo = prophet_data[prophet_data['store_dept'] == combination]
        combo = combo.rename(columns={'Date':'ds', 'Weekly_Sales' : 'y'}) #prophen require ds and y
        
        combo_train = combo[combo['ds'] < min_test_time]
        combo_test = combo[(combo['ds'] >= min_test_time) & (combo['ds'] <= max_test_time)]
        if combo_train.empty or combo_test.empty:
            print(f'Skip {combination} due to lack of data')
            continue
        
        model = Prophet(daily_seasonality=False,weekly_seasonality=True, yearly_seasonality=True, seasonality_mode='multiplicative', changepoint_prior_scale=0.5)
        
        for reg in [ 'IsHoliday_True', 'Temperature', 'Fuel_Price', 'total_markdown']:
            model.add_regressor(reg)
        
        try:
            model.fit(combo_train)
        except Exception as e:
            print(f'Fail to train {combination} : {e}')
            continue
        
        # test
        future = combo_test[['ds', 'Size', 'Type_encoded', 'IsHoliday_True', 'Temperature', 'Fuel_Price', 'total_markdown', 'avg_markdown', 'max_markdown']]
        forecast = model.predict(future)
        
        forecast = forecast[['ds', 'yhat', 'yhat_upper', 'yhat_lower']].merge(combo_test[['ds', 'y']], on=['ds'])
        
        prophet_model[combination] = model
        prophet_predictions[combination] = forecast
        
        # evaluate
        mae = mean_absolute_error(forecast['y'], forecast['yhat'])
        rmse = np.sqrt(mean_squared_error(forecast['y'], forecast['yhat']))
        wampe_metrics = WeightedMeanAbsolutePercentageError()
        y_pred = torch.tensor(forecast['yhat'].to_numpy(), dtype=torch.float32)
        y = torch.tensor(forecast['y'].to_numpy(), dtype=torch.float32)
        wampe = wampe_metrics(y_pred, y).item()
        
        prophet_metrics[len(prophet_metrics)] = [combo, mae, rmse, wampe]
        
        #overall evaluation
        all_real.extend(forecast['y'])
        all_predict.extend(forecast['yhat'])
        
    
    mean_mae = np.mean(prophet_metrics['mae'])
    mean_rmse = np.mean(prophet_metrics['rmse'])
    ovr_wampe = WeightedMeanAbsolutePercentageError(torch.from_numpy(np.array(all_real)), torch.from_numpy(np.array(all_predict)))
    
    return prophet_model, prophet_predictions, (mean_mae, mean_rmse, ovr_wampe)


prophet_model, prophet_predictions, (mean_mae, mean_rmse, ovr_wampe) = build_prophet_model(prophet_data,  test)
    
        
        
        
        
        
        
        
        
        
        
    
    

Build prophet model for  store_10_dept_1
Build prophet model for  store_10_dept_10
Build prophet model for  store_10_dept_11
Build prophet model for  store_10_dept_12
Build prophet model for  store_10_dept_13
Build prophet model for  store_10_dept_14
Build prophet model for  store_10_dept_16
Build prophet model for  store_10_dept_17
Build prophet model for  store_10_dept_18
Build prophet model for  store_10_dept_19
Build prophet model for  store_10_dept_2
Build prophet model for  store_10_dept_20
Build prophet model for  store_10_dept_21
Build prophet model for  store_10_dept_22
Build prophet model for  store_10_dept_23
Build prophet model for  store_10_dept_24
Build prophet model for  store_10_dept_25
Build prophet model for  store_10_dept_26
Build prophet model for  store_10_dept_27
Build prophet model for  store_10_dept_28
Build prophet model for  store_10_dept_29
Build prophet model for  store_10_dept_3
Build prophet model for  store_10_dept_30
Build prophet model for  store_10_dep

ERROR:cmdstanpy:Chain [1] error: code '1' Operation not permitted


Build prophet model for  store_11_dept_1
Build prophet model for  store_11_dept_10


ERROR:cmdstanpy:Chain [1] error: code '1' Operation not permitted


Build prophet model for  store_11_dept_11


...

INFO:prophet:n_changepoints greater than number of observations. Using 1.


Build prophet model for  store_17_dept_47
Skip  store_17_dept_47 due to lack of data
Build prophet model for  store_35_dept_96


INFO:prophet:n_changepoints greater than number of observations. Using 0.
INFO:prophet:n_changepoints greater than number of observations. Using 1.


Build prophet model for  store_30_dept_99
Build prophet model for  store_36_dept_32
Build prophet model for  store_25_dept_48
Skip  store_25_dept_48 due to lack of data
Build prophet model for  store_42_dept_26
Skip  store_42_dept_26 due to lack of data
Build prophet model for  store_44_dept_49
Skip  store_44_dept_49 due to lack of data
Build prophet model for  store_22_dept_96
Skip  store_22_dept_96 due to lack of data
Build prophet model for  store_38_dept_55
Skip  store_38_dept_55 due to lack of data
Build prophet model for  store_19_dept_39
Skip  store_19_dept_39 due to lack of data


KeyError: 'rmse'

In [21]:
print(
    f"Prophet Model Results:\nMAE: {mean_mae:.2f} | RMSE: {mean_rmse:.2f} | WAPE: {ovr_wampe:.2f}%"
)

NameError: name 'mean_mae' is not defined